# B3 — Uncertainty, and the mechanism call for every cancer variant

**Question.** For each of the 6,884 atlas variants, can we say *with error control* that it
destabilises the domain?

**What it does.**
- **Normalised split-conformal intervals** from the held-out SH3 family: half calibrates the
  90 % quantile of |error| / (ensemble SD + 0.05), the other half measures coverage.
- A variant is **destabilising only if the lower end of its interval exceeds 1 kcal/mol**.
- The binding axis is **zero-shot ESM-2 rank-matched** to the measured GRB2-SH3 binding scale,
  because Part C shows the trained binding head cannot beat it (read from C2's decision file).
  It carries **no binding claim**.
- Diagnostics: coverage at several nominal levels (calibration curve) and accuracy of the most
  confident predictions (rejection curve). The old ensemble-spread "confidence" filter is shown
  as a documented failure.

**Inputs:** B1 predictions, A2 ddPCA features, C2 `binding_decision.json`. **Runtime:** under a minute.
**Outputs:** `atlas_predictions.parquet`, `calibration_curve.csv`, `rejection_curve.csv`, `RESULTS_B3.md`.
Explanation: `docs/B3_uncertainty_and_calls.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'B3', 'Uncertainty and mechanism calls', resume=RESUME)
print(C.env_info())

In [ ]:
T = pd.read_parquet(R.res('B1', 'heldout_predictions.parquet'))
A = pd.read_parquet(R.res('B1', 'atlas_fold_predictions.parquet'))
DD = pd.read_parquet(R.res('A2', 'features_ddpca.parquet'))
DEC = json.load(open(R.res('C2', 'binding_decision.json')))
print('binding decision from C2:', DEC)
if DEC['head_beats_esm']:
    raise RuntimeError('C2 found a trained binding head that beats ESM-2 on GRB2-SH3, unlike the original run. '
                       'Investigate before making any atlas call.')
pte, sdte = T.pred_c10.values, T.sd_c10.values
yte = T.ddG_fold.values.astype(np.float32); err = np.abs(yte - pte)
perm = np.random.default_rng(0).permutation(len(T)); cal, val = perm[:len(T) // 2], perm[len(T) // 2:]
EPS = 0.05
def conformal_q(e, sd, alpha):
    s = np.sort(e / (sd + EPS)); return float(s[min(int(np.ceil((len(s) + 1) * (1 - alpha))), len(s)) - 1])
Q90 = conformal_q(err[cal], sdte[cal], 0.10)
COV = float(np.mean(np.abs(yte[val] - pte[val]) <= Q90 * (sdte[val] + EPS)))
SD_ERR = C.spearman(sdte, err)
print(f'q90 {Q90:.4f} | coverage {COV:.2%} (target 90%) | corr(ensemble SD, |error|) {SD_ERR:+.3f}')

## Calibration and rejection curves (diagnostics)

In [ ]:
CAL = pd.DataFrame([dict(nominal=1 - a, quantile=conformal_q(err[cal], sdte[cal], a),
                         empirical=float(np.mean(np.abs(yte[val] - pte[val]) <= conformal_q(err[cal], sdte[cal], a) * (sdte[val] + EPS))),
                         median_halfwidth=float(np.median(conformal_q(err[cal], sdte[cal], a) * (sdte[val] + EPS))))
                    for a in (0.5, 0.4, 0.3, 0.2, 0.1, 0.05)])
order = val[np.argsort(sdte[val])]
REJ = pd.DataFrame([dict(kept_most_confident=f'{int(f*100)}%', n=int(len(order) * f),
                         rho=C.spearman(yte[order[:int(len(order) * f)]], pte[order[:int(len(order) * f)]]),
                         mae=float(np.mean(err[order[:int(len(order) * f)]])))
                    for f in (0.05, 0.1, 0.2, 0.5, 1.0)])
CAL.to_csv(R.out('calibration_curve.csv'), index=False); REJ.to_csv(R.out('rejection_curve.csv'), index=False)
# stratified coverage (roadmap I1): is the global 90% honest in every subgroup? The held-out MegaScale
# SH3 variants carry no structure, so strata are predicted-effect bins and domains, not burial.
cov_v = np.abs(yte[val] - pte[val]) <= Q90 * (sdte[val] + EPS)
Tv = T.iloc[val].assign(covered=cov_v, halfwidth=Q90 * (sdte[val] + EPS),
                        pred_bin=pd.cut(pte[val], [-np.inf, 0, 1, 2, np.inf], labels=['<0', '0-1', '1-2', '>2 kcal/mol']))
STRAT = pd.concat([Tv.groupby('pred_bin', observed=True).agg(n=('covered', 'size'), coverage=('covered', 'mean'),
                                                            median_halfwidth=('halfwidth', 'median')).reset_index()
                     .rename(columns={'pred_bin': 'stratum'}).assign(by='predicted ΔΔG'),
                   Tv.groupby('gene').agg(n=('covered', 'size'), coverage=('covered', 'mean'),
                                          median_halfwidth=('halfwidth', 'median')).reset_index()
                     .rename(columns={'gene': 'stratum'}).assign(by='domain (gene)')])[['by', 'stratum', 'n', 'coverage', 'median_halfwidth']]
STRAT.to_csv(R.out('stratified_coverage.csv'), index=False); print(STRAT.round(3).to_string(index=False))
print(CAL.round(3).to_string(index=False)); print(REJ.round(3).to_string(index=False))

## The calls

In [ ]:
A['fold_halfwidth'] = Q90 * (A.sd_fold.values + EPS); A['fold_lo'] = A.pred_ddG_fold - A.fold_halfwidth
B = DD[DD.ddG_bind.notna() & DD.binding_reliable.eq(True)]
g = B[B.domain_key == 'GRB2-SH3'].ddG_bind.values
r = (-A.esm_score.values).argsort().argsort() / max(1, len(A) - 1)
A['pred_ddG_bind'] = np.quantile(np.sort(g), np.clip(r, 0, 1)); A['bind_source'] = 'esm_rank_matched'
A['bind_lo'] = A.pred_ddG_bind - float(np.quantile(np.abs(g - np.median(g)), 0.90))
f, b = A.fold_lo >= 1.0, A.bind_lo >= 1.0
A['mechanism'] = np.select([f & b, f & ~b, ~f & b], ['both', 'destabilising', 'rewiring'], 'tolerated')
A.to_parquet(R.out('atlas_predictions.parquet'), index=False)
MC = A.mechanism.value_counts()
AXIS = C.spearman(A.pred_ddG_fold, A.pred_ddG_bind)
print(MC.to_string()); print(f'point-estimate calls (pred >= 1, no interval): {int((A.pred_ddG_fold >= 1).sum()):,}')

## Results

In [ ]:
for k, v in [('conformal_q90', Q90), ('conformal_coverage', COV), ('sd_vs_error_rho', SD_ERR),
             *[(f'n_{m}', int(MC.get(m, 0))) for m in ('tolerated', 'destabilising', 'rewiring', 'both')],
             ('point_estimate_destabilising', int((A.pred_ddG_fold >= 1).sum())),
             ('old_filter_pass_fraction', float((A.sd_fold < 0.5).mean())),
             ('axis_rank_correlation', AXIS), ('axis_correlation_is_valid_test', False),
             ('median_fold_halfwidth', float(A.fold_halfwidth.median()))]:
    R.metric(k, v)
R.section('Mechanism calls', 'Destabilising = lower interval bound ≥ 1 kcal/mol. "Rewiring" rests on rank-matched ESM-2 '
          'and is reported, not claimed.', MC.rename_axis('mechanism').reset_index(name='variants'))
R.section('Calibration curve (validation half)', 'Nominal against empirical coverage.', CAL, '{:.3f}')
R.section('Rejection curve (validation half)', 'Accuracy of the most confident predictions, ranked by ensemble SD.', REJ, '{:.3f}')
R.section('Stratified coverage (validation half)', 'The 90% target is global; this shows where it is met and where not. '
          'Burial cannot be used as a stratum here: the held-out MegaScale variants carry no structure.', STRAT, '{:.3f}')
R.metric('coverage_min_domain', float(STRAT[STRAT.by == 'domain (gene)'].coverage.min()))
R.metric('coverage_max_domain', float(STRAT[STRAT.by == 'domain (gene)'].coverage.max()))
R.section('Why the old confidence filter was dropped', f"The original Stage 3 called a prediction confident when the "
          f"five seeds agreed (SD < 0.5). Seed spread correlates only ρ {SD_ERR:+.3f} with the actual error, and "
          f"{(A.sd_fold < 0.5).mean():.1%} of atlas variants pass it: agreement between seeds is reproducibility, not accuracy.")
R.section('Axis correlation', f"ρ(predicted fold, predicted bind) = {AXIS:+.3f}. **Not a valid test** of whether a separate "
          "rewiring class exists: both axes derive from ESM-2 by construction.")
R.finish([f"Conformal q90 {Q90:.3f}; empirical coverage {COV:.1%} against a 90% target.",
          f"{int(MC.get('destabilising',0)):,} destabilising, {int(MC.get('both',0)):,} both, {int(MC.get('rewiring',0)):,} rewiring, "
          f"{int(MC.get('tolerated',0)):,} tolerated — {int((A.pred_ddG_fold >= 1).sum()):,} point-estimate calls reduced "
          f"to {int(MC.get('destabilising',0) + MC.get('both',0)):,} under error control.",
          'The binding axis is rank-matched zero-shot ESM-2 (Part C); no binding claim is made.'])